In [1]:

import torch
import os


In [4]:
checkpoint_path = "best_model.pth"
checkpoint = torch.load(checkpoint_path, map_location='cpu')
# Assuming 'checkpoint' is already loaded
internal_keys = checkpoint['model'].keys()

# Check for Discriminator-specific identifiers
is_discriminator = any('disc' in k.lower() or 'msd' in k.lower() or 'mpd' in k.lower() for k in internal_keys)

# Check for Generator-specific identifiers
is_generator = any('enc_p' in k or 'flow' in k or 'dec' in k for k in internal_keys)

print(f"Is Discriminator: {is_discriminator}")
print(f"Is Generator: {is_generator}")
print(f"Current Step: {checkpoint['step']}")
print(f"Current Epoch: {checkpoint['epoch']}")

Is Discriminator: True
Is Generator: True
Current Step: 135916
Current Epoch: 36


In [ ]:

def load_checkpoint(checkpoint_path, model, optimizer=None):
    """
    Loads a VITS checkpoint into the model and optimizer.
    """
    assert os.path.isfile(checkpoint_path), f"Checkpoint not found at: {checkpoint_path}"
    
    # Load the file to CPU first to avoid VRAM spikes
    checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
    
    # 1. Load the Model Weights
    # Most VITS checkpoints store weights under the 'model' key
    if 'model' in checkpoint_dict:
        model.load_state_dict(checkpoint_dict['model'])
    else:
        # Fallback for some versions that save state_dict directly
        model.load_state_dict(checkpoint_dict)
        
    # 2. Load the Optimizer (Crucial for resuming training)
    if optimizer is not None and 'optimizer' in checkpoint_dict:
        optimizer.load_state_dict(checkpoint_dict['optimizer'])
        
    # 3. Extract the iteration/step count
    iteration = checkpoint_dict.get('iteration', 1)
    learning_rate = checkpoint_dict.get('learning_rate', None)
    
    print(f"Successfully loaded checkpoint: {checkpoint_path} (Iteration: {iteration})")
    return model, optimizer, learning_rate, iteration

# --- Usage Example ---

# Define your paths
path_g = "logs/my_model/G_140000.pth"
path_d = "logs/my_model/D_140000.pth"

# Load Generator
net_g, optim_g, lr, start_iter = load_checkpoint(path_g, net_g, optim_g)

# Load Discriminator (use same iteration/lr logic)
net_d, optim_d, _, _ = load_checkpoint(path_d, net_d, optim_d)